# Lab 13: IoT Network Anomaly Detection

**Task:** detect abnormal network traffic from IoT devices (smart speakers, cameras) on a home network.

**Dataset:** IoTID20 (Kaggle: https://www.kaggle.com/datasets/rohulaminlabid/iotid20-dataset).

> **The data is not in this repository: `IoT Network Intrusion Dataset.csv` is about 300 MB, too big for git.** Download it from the link above and put the CSV in this lab's `data/` folder.

625,783 network **flows** (a flow is one conversation between two devices: all the packets between two addresses and ports over some time), each described by 80 statistics: packet counts and sizes, durations, gaps between packets, TCP flags. Each flow is labelled Normal or as an attack: Mirai botnet, port scans, denial of service (DoS), and man-in-the-middle (ARP spoofing).

**What we will do:**
1. Select traffic features and clean them (missing and infinite values, useless columns), then standardise.
2. **Unsupervised anomaly detection**: cluster *normal* traffic with K-means, score every flow by its distance to the nearest normal cluster centre, and flag the most distant ones, comparing thresholds.
3. **Supervised version**: logistic regression and a random forest trained on the labels.
4. Compare the two approaches, show the distance distributions and the most anomalous flows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 14)
flows = pd.read_csv("data/IoT Network Intrusion Dataset.csv")
print(flows.shape)
print(flows["Label"].value_counts().to_dict())
print(flows["Cat"].value_counts().to_dict())

Note the proportions: **94% of the flows are attacks**. The researchers ran attacks on purpose to collect them; on a real network attacks are rare. For anomaly detection this changes nothing: we describe *normal* traffic and measure how far each flow is from it. For evaluation we'll keep in mind that precision depends heavily on the share of attacks.

We take a random sample of 120,000 flows (stratified by category) to keep the lab fast.

In [ ]:
flows = flows.groupby("Cat", group_keys=False).sample(frac=120_000 / len(flows), random_state=0).reset_index(drop=True)
flows["attack"] = (flows["Label"] == "Anomaly").astype(int)
print(flows.shape, "| attack share:", round(flows["attack"].mean(), 3))

## 1. Selecting and cleaning features

- **Identifiers are not features.** `Flow_ID`, the IP addresses, the ports and the timestamp identify *which* machines talked *when*. In this testbed the attacker machines have fixed addresses, so a model could learn "this IP is the attacker", useless on any other network. We use only the traffic statistics.
- **Infinite values.** Rates such as bytes per second divide by the flow duration, which is 0 for some flows; the result is infinite. We turn infinities into missing values and fill them with the median of the normal training traffic.
- **Constant columns** (the same value in every flow) carry no information; we drop them.

In [ ]:
id_cols = ["Flow_ID", "Src_IP", "Src_Port", "Dst_IP", "Dst_Port", "Timestamp"]
label_cols = ["Label", "Cat", "Sub_Cat", "attack"]
features = [c for c in flows.columns if c not in id_cols + label_cols]
X_all = flows[features].replace([np.inf, -np.inf], np.nan)
print("infinite values:", int(np.isinf(flows[features]).sum().sum()), "| columns with them:", X_all.columns[X_all.isna().any()].tolist())
constant = [c for c in features if X_all[c].nunique() <= 1]
features = [c for c in features if c not in constant]
print("constant columns dropped:", constant)
print(len(features), "features left, e.g.", features[:8])

### Train/test split
Supervised and unsupervised models get the same split. The unsupervised model only ever looks at the **normal** flows of the training set.

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(flows, test_size=0.3, stratify=flows["Cat"], random_state=0)
normal_train = train[train["attack"] == 0]
medians = normal_train[features].replace([np.inf, -np.inf], np.nan).median()

def prepare(df):
    x = df[features].replace([np.inf, -np.inf], np.nan).fillna(medians)
    return np.sign(x) * np.log1p(x.abs())        # counts, bytes and durations span many orders of magnitude

x_train, x_test, x_normal = prepare(train), prepare(test), prepare(normal_train)
print("training flows:", len(train), "| of which normal:", len(normal_train), "| test flows:", len(test))

Most of these statistics are extremely skewed (a flow can last a microsecond or an hour; carry 40 bytes or 40 megabytes). On the raw scale, a few huge flows would dominate every distance, so we take a signed log, $\text{sign}(x)\log(1 + |x|)$, and then **standardise** each feature (mean 0, standard deviation 1), learning the means and standard deviations from normal training traffic only.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(x_normal)
z_normal, z_train, z_test = scaler.transform(x_normal), scaler.transform(x_train), scaler.transform(x_test)

## 2. Unsupervised: K-means on normal traffic

Normal traffic is not one kind of thing: video streaming, DNS lookups and device heartbeats look very different. K-means finds the main kinds as clusters. A new flow that is close to one of these cluster centres looks like some known normal behaviour; a flow far from all of them is unusual. Its **distance to the nearest centre** is the anomaly score.

How many clusters? We check the SSE (elbow) curve on the normal flows:

In [ ]:
from sklearn.cluster import KMeans

sse = {k: KMeans(n_clusters=k, n_init=5, random_state=0).fit(z_normal).inertia_ for k in [2, 4, 6, 8, 10, 15, 20]}
plt.plot(list(sse), list(sse.values()), "o-"); plt.xlabel("k"); plt.ylabel("SSE"); plt.title("K-means on normal traffic"); plt.show()

In [ ]:
km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(z_normal)
dist_normal = km.transform(z_normal).min(axis=1)      # distance of each normal training flow to its nearest centre
dist_test = km.transform(z_test).min(axis=1)
print("cluster sizes (normal training flows):", np.bincount(km.labels_))

The curve bends gradually; we take $k = 10$, enough to capture the main kinds of normal traffic. Notice that three of the clusters are tiny (1, 2 and 14 flows): K-means gave rare, unusual normal flows clusters of their own. Flows near those will count as "normal" too; with a larger $k$ this effect grows, which is one reason not to make $k$ too large for anomaly detection.

### Choosing the threshold
"Flag the top X%" needs a reference. We set the threshold at a high percentile of the distances of **normal** training flows: at the 95th percentile, by construction about 5% of normal traffic is flagged (the false-alarm rate we accept).

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

y_test = test["attack"].values
rows = []
for pct in [80, 90, 95, 99, 99.9]:
    thr = np.percentile(dist_normal, pct)
    flag = (dist_test > thr).astype(int)
    rows.append({"threshold": f"{pct}th percentile of normal", "distance": thr, "flagged %": 100 * flag.mean(),
                 "recall (attacks caught)": recall_score(y_test, flag), "false alarm rate": flag[y_test == 0].mean(),
                 "precision": precision_score(y_test, flag)})
thresholds = pd.DataFrame(rows)
print("ROC AUC of the distance score:", round(roc_auc_score(y_test, dist_test), 4))
thresholds.round(3)

### Anomaly scoring plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(0, np.percentile(dist_test, 99.5), 120)
for cat in ["Normal", "Mirai", "Scan", "DoS", "MITM ARP Spoofing"]:
    ax.hist(dist_test[test["Cat"].values == cat], bins=bins, alpha=0.5, density=True, label=cat)
ax.axvline(np.percentile(dist_normal, 95), color="k", ls="--", label="95th percentile of normal")
ax.set_xlabel("distance to nearest normal cluster centre"); ax.set_ylabel("density"); ax.legend(); plt.show()

Which attacks does the distance catch, at the 95th-percentile threshold?

In [ ]:
thr95 = np.percentile(dist_normal, 95)
test_view = test[["Cat", "Sub_Cat"]].assign(distance=dist_test, flagged=dist_test > thr95)
test_view.groupby("Sub_Cat")["flagged"].agg(["size", "mean"]).rename(columns={"size": "flows", "mean": "share flagged"}).sort_values("share flagged").round(3)

### The most anomalous flows

In [ ]:
show = ["Protocol", "Flow_Duration", "Tot_Fwd_Pkts", "Tot_Bwd_Pkts", "TotLen_Bwd_Pkts", "Flow_IAT_Mean", "Pkt_Len_Mean", "SYN_Flag_Cnt", "ACK_Flag_Cnt"]
top = test_view.sort_values("distance", ascending=False).head(10)
pd.concat([top[["Sub_Cat", "distance"]], test.loc[top.index, show]], axis=1).round(2)

The most anomalous flows are all of one strange kind: **no forward packets at all**, two packets in the other direction, no payload. Almost no normal flow looks like this, so they lie far from every normal centre. Nearly all are attack traffic (Mirai floods, brute force, spoofing); one is labelled normal, a reminder that "unusual" is not the same as "malicious".

Reading the share flagged per attack type (two tables up): the detector catches **all SYN floods** and two thirds of **UDP floods**, which produce flows unlike anything in normal traffic. It catches only 12–14% of **port scans** and **HTTP floods**: each individual scan flow is a short, ordinary-looking connection attempt. A scan is obvious only in *aggregate* (one source contacting hundreds of ports in seconds), which per-flow features can't see. Features that summarise each source's activity over a time window would fix that.

## 3. Supervised: learning from the labels

With labelled attacks, we can train classifiers directly. Logistic regression on the standardised features, and a random forest (trees don't need scaling; we give it the log-transformed features).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

y_train = train["attack"].values
supervised = {
    "logistic regression": (LogisticRegression(max_iter=3000, C=1.0), z_train, z_test),
    "random forest": (RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=0), x_train, x_test),
}
scores = {"K-means distance (unsupervised)": dist_test}
for name, (model, xtr, xte) in supervised.items():
    model.fit(xtr, y_train)
    scores[name] = model.predict_proba(xte)[:, 1]

## 4. Comparison

For a fair comparison every method gets the same false-alarm budget: the threshold is set so that 5% of *normal* test flows are flagged, and we compare how many attacks each catches. ROC AUC compares them over all thresholds.

In [ ]:
rows = []
for name, s in scores.items():
    thr = np.percentile(s[y_test == 0], 95)
    flag = (s > thr).astype(int)
    rows.append({"method": name, "ROC AUC": roc_auc_score(y_test, s), "recall at 5% false alarms": recall_score(y_test, flag),
                 "F1": f1_score(y_test, flag)})
    by_cat = pd.Series(flag).groupby(test["Cat"].values).mean()
    rows[-1].update({f"caught: {c}": by_cat[c] for c in ["Mirai", "Scan", "DoS", "MITM ARP Spoofing"]})
pd.DataFrame(rows).set_index("method").round(3)

### Unsupervised vs supervised

- **The supervised models are far better** at detecting these attacks: the random forest catches 97% at a 5% false-alarm rate (ROC AUC 0.99), logistic regression 87%, the K-means distance 39% (ROC AUC 0.85). With labelled examples of every attack, the models learn exactly what each one looks like on this network, including attacks whose individual flows look ordinary (scans: 90% for the forest, 8% for the distance score).
- **The unsupervised detector** learned only what *normal* looks like, so it finds attacks that make individual flows unusual (SYN and UDP floods) and misses attacks that hide among normal-looking flows (scans, HTTP floods, much of the ARP spoofing).
- **But the unsupervised approach has two big advantages** in real use: it needs no labelled attacks (expensive to get and always incomplete), and it can flag *new* kinds of attack that no labelled dataset contains; a supervised model only recognises what it has seen. In practice both are combined: a supervised model for known attacks, an anomaly detector for the unknown, and a person reviewing what is flagged.
- **Precision depends on the attack share.** Here 94% of the flows are attacks, so almost every alert is a real attack (precision 0.98–0.99 in the threshold table). On a real network where 0.1% of flows are malicious, the same 5% false-alarm rate would produce about 50 false alarms per real attack: the threshold would have to be much stricter, at the cost of recall.